# Loading Models

This notebook serves to download model checkpoint files hosted from the [Microsoft Aurora Hugging Face repository](https://huggingface.co/microsoft/aurora) and register these as Azure Machine Learning model assets.

This notebook does not require GPU capable compute. If running jobs remotely, it is recommend to run it in the Notebooks area of the AML Studio UI to mitigate download and upload. 

In [ ]:
%pip install huggingface-hub

In [ ]:
import sys
from pathlib import Path

from azure.ai.ml.entities import Model
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import DefaultAzureCredential
from huggingface_hub import hf_hub_download

# insert parent directory to path for proper absolute local imports
sys.path.insert(0, str(Path.cwd().parent.parent.resolve()))
from setup.common.constants import HF_REPOSITORY, MODEL_ASSET_NAME, MODEL_FILENAME
from setup.common.utils import create_mlclient, get_latest_asset


Download the model checkpoint file to the local `huggingface-hub` cache directory on the compute instance.

#### Notes

See [setup/common/constants.py](../../setup/common/constants.py#10). to change the checkpoint to download. This is not recommended - workshop code is designed for use with Aurora 0.25 Pre-trained.

In [ ]:
path = hf_hub_download(repo_id=HF_REPOSITORY, filename=MODEL_FILENAME)
print(f"Model downloaded to local cache: {path}")

Obtain and create necessary environment parameters and Azure interface objects and register the model as an AML model asset.

#### Notes

Skip this cell if running the workshop end-to-end locally. 

In [ ]:
az_cred = DefaultAzureCredential()
ml_client = create_mlclient(local=False)
try:
    asset = get_latest_asset(ml_client.models, MODEL_ASSET_NAME)
    if asset.version is None:
        msg = f"Version is None for asset: {MODEL_ASSET_NAME}"
        raise ValueError(msg)
    new = int(asset.version) + 1
except ResourceNotFoundError:
    new = 1

model_asset = Model(name=MODEL_ASSET_NAME, version=str(new), path=path)
ml_client.models.create_or_update(model_asset)
print(
    f"Created or updated asset: name={model_asset.name}, "
    f"version={model_asset.version}, path={model_asset.path}",
)